In [ ]:
# Path configuration
from pathlib import Path

PROJECT_ROOT = Path("..")  # Repository root.
BASELINE = PROJECT_ROOT / "data"
FIRM_DATA = PROJECT_ROOT
PATSTAT = PROJECT_ROOT / "patstat"
PATENT_TEXT = PROJECT_ROOT / "patent_text"
SAMPLEDATA = PROJECT_ROOT / "sampledata"
DATA = Path("data")          # Generated revision intermediates.
OUTPUTS = Path("outputs")    # Tables, figures and saved comparison results.
SEED = 42


# Corporate-group screening results and Fig. 4

Run from `revision_notebooks/` after notebook 11, with paths set in the first cell. Read its `top_candidates.parquet`, `observed_judgments.csv` and `top_candidate_judgments.csv` from `DATA/alliance/`, and `fig4a_statistics.csv` and `fig4b_statistics.csv` from `OUTPUTS/alliance/`.

Count corporate-group classifications, calculate fractional organization-type shares and country flows, then draw Fig. 4. The outputs also supply the SI S6 table on co-assignments between companies in different countries. Notebook 12 fits the alliance regressions.


In [ ]:
from collections import Counter
from fractions import Fraction
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go


intermediate = DATA / 'alliance'
output = OUTPUTS / 'alliance'
output.mkdir(parents=True, exist_ok=True)

CLASSES = ['different_group', 'same_group', 'uncertain']
ORDER = ['COMPANY', 'UNIVERSITY', 'GOV', 'NON-PROFIT', 'HOSPITAL', 'UNKNOWN']
LABELS = ['[1-2]', '[3-5]', '[6-10]', '[11-50]', '[51-100]', '>100']

## Load the candidates and judgments saved by notebook 11


In [ ]:
top = pd.read_parquet(intermediate / 'top_candidates.parquet')
observed_judgments = pd.read_csv(intermediate / 'observed_judgments.csv', keep_default_na=False)
top_judgments = pd.read_csv(intermediate / 'top_candidate_judgments.csv', keep_default_na=False)

## Fractional organization types

Each selected pair contributes one total weight, split evenly among its type labels. Collapse NON-PROFIT into GOV only when GOV is present and the two labels co-occur in both directions throughout this selected population. Counts and denominators come from candidate metadata independently of ownership judgments.

In [ ]:
def fractional_types(raw_labels):
    label_sets = [set(x.upper().split()) or {'UNKNOWN'} for x in raw_labels]
    collapse = any('GOV' in x for x in label_sets) and all(
        ('GOV' in x) == ('NON-PROFIT' in x) for x in label_sets)
    counts = Counter()
    for labels in label_sets:
        retained = labels - {'NON-PROFIT'} if collapse else labels
        for label in retained:
            counts[label] += Fraction(1, len(retained))
    return counts, collapse

In [ ]:
# Figure denominators and labels come from selected-candidate metadata, independently of judgments.
top = top.copy()
top['supplyctry'] = top.supplyctry.str.strip()
top['demandctry'] = top.demandctry.str.strip()
known = top.supplyctry.ne('') & top.demandctry.ne('')
cross = known & top.supplyctry.ne(top.demandctry)
counts, collapse = fractional_types(top.supplytype)
fractions = pd.DataFrame([{'category':label, 'fractional_count':float(counts[label]),
    'percent':100 * float(counts[label]) / len(top), 'denominator':len(top)}
    for label in ORDER if label in counts])
fractions.to_csv(output / 'fractional_types.csv', index=False)

fractions

## Country flows

Use stripped, nonblank country values and retain domestic flows. Group countries outside US/JP/KR/DE/CN into Other countries for Fig. 4d. Countries identify the firms' main R&D locations.


In [ ]:
flows = top.loc[known].groupby(['demandctry','supplyctry']).agg(
    pairs=('supply','size')).reset_index()
flows.to_csv(output / 'country_flows.csv', index=False)
major = ['JP','CN','US','DE','KR']
grouped = flows.copy()
for col in ['demandctry','supplyctry']:
    grouped[col] = grouped[col].where(grouped[col].isin(major), 'other')
grouped.groupby(['demandctry','supplyctry'],as_index=False).pairs.sum().to_csv(
    output / 'country_flows_major.csv', index=False)
country = top.assign(country_observed=known, cross_border=cross).groupby('demandctry').agg(
    pairs=('demand','size'), country_observed=('country_observed','sum'), cross_border=('cross_border','sum'))
country['cross_border_share'] = country.cross_border / country.country_observed
country.to_csv(output / 'country_candidate_counts.csv')

## Count corporate-group classifications and sample sizes

Join selected-candidate classifications by focal and candidate PSN IDs. Count classifications separately for observed co-assignments, all selected firm–candidate pairs, and selected pairs with different known countries. Report the total number of pairs in each sample alongside the classification counts.


In [ ]:
# Join candidate judgments by directed focal/candidate IDs.
judged = top[['supply','demand']].assign(cross_border=cross).merge(top_judgments,
    left_on=['supply','demand'], right_on=['supply_psn_id','demand_psn_id'], suffixes=('', '_saved'))
cohorts = [('observed_company_cross_country', observed_judgments),
    ('top_candidates_all', judged), ('top_candidates_cross_border', judged.loc[judged.cross_border])]
rows = []
for name, frame in cohorts:
    for label in CLASSES:
        n = int(frame.classification.eq(label).sum())
        rows.append(dict(cohort=name, classification=label, count=n, denominator=len(frame), percent=100*n/len(frame)))
pd.DataFrame(rows).to_csv(output / 'ownership_counts.csv', index=False)
checks = dict(selected_pairs=len(top), selected_organizations=int(top.supply.nunique()),
    country_observed=int(known.sum()), cross_border=int(cross.sum()),
    observed_audited_pairs=len(observed_judgments),
    gov_nonprofit_collapsed=collapse, fractional_total=str(sum(counts.values())))
(output / 'counts.json').write_text(json.dumps(checks, indent=2)+'\n')

checks

## Fig. 4a/b: complementarity and alliance frequency

Load the statistics calculated by notebook 11 after excluding pairs of a firm with itself. The files contain means, standard deviations, standard errors and intervals of 1.96 times the standard error.


In [ ]:
for panel, labels, colors, ylabel, xlabel in [
    ('a', ['No','Yes'], ['grey','#107010'], 'GBB complementarity', 'Observed green alliance'),
    ('b', LABELS, ['#107010']*6, 'Probability of green alliance', 'GBB complementarity rank')]:
    frame = pd.read_csv(output / f'fig4{panel}_statistics.csv')
    fig, ax = plt.subplots(figsize=(5,3))
    x = np.arange(len(frame))
    ax.bar(x, frame['mean'], color=colors)
    ax.errorbar(x, frame['mean'], yerr=frame.ci95, fmt='none', c='black')
    ax.set(xticks=x, xticklabels=labels, ylabel=ylabel, xlabel=xlabel)
    fig.tight_layout()
    fig.savefig(output / f'fig4{panel}.pdf')
    plt.show()

## Fig. 4c: fractional organization shares

In [ ]:
fractions = pd.read_csv(output / 'fractional_types.csv')
fig, ax = plt.subplots(figsize=(5,3))
bars = ax.barh(fractions.category.replace({'GOV':'GOVERNMENT'}), fractions.percent, color='grey')
ax.bar_label(bars, labels=[f'{p:.1f}%' for p in fractions.percent], padding=3)
ax.invert_yaxis()
ax.set(xlabel='Fractional share of top candidates (%)', xlim=(0,80))
fig.tight_layout()
fig.savefig(output / 'fig4c.pdf')
plt.show()

## Fig. 4d: focal-to-candidate country flows

Save a standalone interactive HTML alongside the three PDF panels.

In [ ]:
flows = pd.read_csv(output / 'country_flows_major.csv')
codes = ['US','JP','KR','DE','CN','other']
names = ['United States','Japan','South Korea','Germany','China','Other countries']
chart = go.Sankey(arrangement='snap',
    node=dict(label=names+names, x=[0]*6+[1]*6),
    link=dict(source=[codes.index(c) for c in flows.demandctry],
              target=[6+codes.index(c) for c in flows.supplyctry], value=flows.pairs))
fig = go.Figure(chart)
fig.update_layout(title='Focal firms (left) and most complementary candidates (right)',
                  width=1000, height=650)
fig.write_html(output / 'fig4d_country_flows.html', include_plotlyjs=True)

fig